# RAG Bot 

The RAG bot system to answer the question about Ethnicity in Thailand.

## Setting Up   

### Install Libraries & Dependencies  

In [13]:
%pip install python-dotenv easyocr pypdf  

Note: you may need to restart the kernel to use updated packages.


### Import Libraries & Dependencies  

In [14]:
import os, re, json, glob   
import time  
import requests  
from dotenv import load_dotenv   

### Setting Environments  

In [15]:
load_dotenv()

True

In [16]:
api_key = os.getenv("THAILLM_API_KEY") 

if api_key:
  print(f"[LOG] Success: Get API Key success.")
else:
  print(f"[LOG] Fail: Cannot Get API Key.")  

[LOG] Success: Get API Key success.


## LLM Implementation  

In [17]:
def ask(messages: str, model="typhoon", max_retries=5):
  url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"  
  headers = {
    "Content-Type": "application/json",
    "apikey": api_key  
  }
  payload = {
    "model": "/model",
    "messages": messages,
    "max_tokens": 2024,
    "temperature": 0 
  }

  for attempt in range(max_retries):
    try:
      resp = requests.post(url=url, headers=headers, json=payload, timeout=120)

      if resp.status_code == 429:
        wait = min(2 ** attempt, 30)
        print(f"Rate Limit, wating {wait} sec(s).")
        time.sleep(wait)
        continue 
      
      resp.raise_for_status()
      text_answer = resp.json()["choices"][0]["message"]["content"].strip() 
      clean_answer = re.sub(r"<think>.*?</think>", "", text_answer, flags=re.DOTALL).strip()
      return clean_answer    
    except requests.exceptions.RequestException as e:
      wait = 2 ** attempt
      print(f"Error: {e}, Retrying in {wait} sec(s).")
      time.sleep(wait)
    
  return None  


In [18]:
def qa(question: str):
  prompt = {"role": "user", "content": question}

  answer_typhoon = ask([prompt], model="typhoon")
  answer_kbtg = ask([prompt], model="kbtg")
  answer_openthaigpt = ask([prompt], model="openthaigpt")
  answer_pathumma = ask([prompt], model="pathumma")

  # JSON Format
  # answer = {
  #   "Typhoon": f"[RAG BOT (Typhoon)] {answer_typhoon}",
  #   "KBTG": f"[RAG BOT (KBTG)] {answer_kbtg}",
  #   "OpenThaiGPT": f"[RAG BOT (OpenThaiGPT)] {answer_openthaigpt}",
  #   "Pathumma": f"[RAG BOT (Pathumma)] {answer_pathumma}"
  # }

  # Markdown Format   
  answer = f"""# Question
{question}

# Answer

## Typhoon
{answer_typhoon}

## KBTG
{answer_kbtg}

## OpenThaiGPT
{answer_openthaigpt}

## Pathumma
{answer_pathumma} 
"""

  return answer   

### LLM Implementation Testing  

In [19]:
output_path = "../output/answer/markdown"
def report(question: str, filename: str, output_path_dir: str = f"{output_path}"):
  # question = "คนไทยเชื้อสายจีนในเมืองหาดใหญ่ ส่วนมากเป็นคนเชื้อสายจีนสายไหน"
  result = qa(question)

  # output = {
  #   "question": question,
  #   "answer": test_response_01
  # }

  # output_path = "../output/answer/markdown"  

  if filename:
    file_name = filename
  else:
    file_name = "_".join(question.split(" "))

  with open(f"{output_path_dir}/{file_name}.md", "w", encoding="utf-8") as file:
    file.write(result)  

In [20]:
file_name = "QA_Hainanese"
if os.path.isfile(f"{output_path}/{file_name}.md"):
  print(f"[LOG] Already have file.")
else:
  report("คนไทยเชื้อสายจีนไหหลำในไทย อาศัยอยู่กันเยอะในจังหวัดอะไรในประเทศไทยบ้าง", filename="QA_Hainanese")  

[LOG] Already have file.


In [21]:
file_name = "QA_Vietnamese"
if os.path.isfile(f"{output_path}/{file_name}.md"):
  print(f"[LOG] Already have file.")
else:  
  report("คนไทยญวน พูดภาษาเวียดนามเหมือนกับที่เวียดนามในปัจจุบันพูดหรือไม่", filename="QA_Vietnamese")    

[LOG] Already have file.


In [22]:
with open("../data/questions/questions.txt", "r", encoding="utf-8") as f:
  questions = f.readlines()

for index, question in enumerate(questions):
  report(question, filename=f"answer_{index+1}", output_path_dir="../output/answer/markdown/question_txt_response/llm_without_rag")

## RAG Implementation  

### PDF OCR to Markdown   

In [23]:
knowledge = "../data/knowledge/pdf"
files = glob.glob(f"{knowledge}/*.pdf") 
print(files)  
print(f"[LOG] Amount of File: {len(files)} files.")  

['../data/knowledge/pdf/ญัฮกุร.pdf', '../data/knowledge/pdf/ลัวะ.pdf', '../data/knowledge/pdf/จีน.pdf', '../data/knowledge/pdf/โอก๋อง.pdf', '../data/knowledge/pdf/มอญ.pdf', '../data/knowledge/pdf/ไทเบิ้ง.pdf', '../data/knowledge/pdf/ลีซู.pdf', '../data/knowledge/pdf/กะซอง.pdf', '../data/knowledge/pdf/คะฉิ่น.pdf', '../data/knowledge/pdf/โพล่ง.pdf', '../data/knowledge/pdf/อูรักลาโวยจ.pdf', '../data/knowledge/pdf/บรู.pdf', '../data/knowledge/pdf/ไทลื้อ.pdf', '../data/knowledge/pdf/ยอง.pdf', '../data/knowledge/pdf/มานิ.pdf', '../data/knowledge/pdf/ซำเร.pdf', '../data/knowledge/pdf/กูย.pdf', '../data/knowledge/pdf/กะเลิง.pdf', '../data/knowledge/pdf/อิ้วเมี่ยน.pdf', '../data/knowledge/pdf/มอแกน.pdf', '../data/knowledge/pdf/ม้ง.pdf', '../data/knowledge/pdf/ไทใหญ่.pdf', '../data/knowledge/pdf/ไทดำ.pdf', '../data/knowledge/pdf/ละว้า.pdf', '../data/knowledge/pdf/มอแกลน.pdf', '../data/knowledge/pdf/ลาหู่.pdf']
[LOG] Amount of File: 26 files.


In [24]:
from pypdf import PdfReader  

for index, file in enumerate(sorted(files)):
  extract_data = ""
  reader = PdfReader(file)
  text = f"\n".join(p.extract_text() for p in reader.pages) 
  extract_data += text 
  with open(f"../data/knowledge/text/{file.split("/")[-1].split(".")[0]}.txt", "w", encoding="utf-8-sig") as f:
    f.write(extract_data)